# Objective of This Notebook

The objective of this notebook is to transform the cleaned financial statement dataset into a business-focused KPI dataset suitable for clustering analysis. KPI engineering will be performed using a configurable KPI definition framework, allowing financial metrics to be calculated dynamically from the underlying financial statement data.

Once the KPI dataset has been generated, the notebook will evaluate data quality, prepare the features for modelling, and apply dimensionality reduction and clustering techniques to identify groups of financially similar companies. The resulting clusters will serve as the foundation for automated peer benchmarking and subsequent business interpretation.

# **1. Notebook Setup**

### **1.1 Imports**

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA

from sklearn.cluster import KMeans

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

### **1.2 Notebook Configuration**

In [2]:
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

# Plot settings
plt.style.use("default")

plt.rcParams["figure.figsize"] = (12, 6)

np.random.seed(RANDOM_STATE)

### **1.3 Paths**

In [3]:
PROJECT_ROOT = Path().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"

SRC_DIR = PROJECT_ROOT / "src"

# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

### **1.4 Project Imports**

In [4]:
from src.features.kpi_engine import (
    load_kpi_definitions,
    validate_kpi_definitions,
    calculate_kpis,
)

# **2. Dataset Overview**

### **2.1 Load Clean Dataset**

Here we load the final output from Notebook #2

In [5]:
financials_df = pd.read_excel(
    INTERIM_DATA_DIR / "clean_financials_with_metadata.xlsx"
)

print(f"Dataset shape: {financials_df.shape}")

financials_df.head()

Dataset shape: (4576, 222)


,Symbol,Company Name,Exchange,Sector,Industry,Employees,MarketCap,EnterpriseValue,Tax Effect Of Unusual Items,Tax Rate For Calcs,Normalized EBITDA,Total Unusual Items,Total Unusual Items Excluding Goodwill,Net Income From Continuing Operation Net Minority Interest,Reconciled Depreciation,Reconciled Cost Of Revenue,EBITDA,EBIT,Net Interest Income,Interest Expense,Interest Income,Normalized Income,Net Income From Continuing And Discontinued Operation,Total Expenses,Total Operating Income As Reported,Diluted Average Shares,Basic Average Shares,Diluted EPS,Basic EPS,Diluted NI Availto Com Stockholders,Net Income Common Stockholders,Net Income,Net Income Including Noncontrolling Interests,Net Income Continuous Operations,Tax Provision,Pretax Income,Other Income Expense,Other Non Operating Income Expenses,Special Income Charges,Other Special Charges,Impairment Of Capital Assets,Gain On Sale Of Security,Net Non Operating Interest Income Expense,Interest Expense Non Operating,Interest Income Non Operating,Operating Income,Operating Expense,Depreciation Amortization Depletion Income Statement,Depreciation And Amortization In Income Statement,Research And Development,Selling General And Administration,Selling And Marketing Expense,General And Administrative Expense,Other Gand A,Gross Profit,Cost Of Revenue,Total Revenue,Operating Revenue,Treasury Shares Number,Ordinary Shares Number,Share Issued,Net Debt,Total Debt,Tangible Book Value,Invested Capital,Working Capital,Net Tangible Assets,Capital Lease Obligations,Common Stock Equity,Total Capitalization,Total Equity Gross Minority Interest,Stockholders Equity,Gains Losses Not Affecting Retained Earnings,Other Equity Adjustments,Retained Earnings,Additional Paid In Capital,Capital Stock,Common Stock,Preferred Stock,Total Liabilities Net Minority Interest,Total Non Current Liabilities Net Minority Interest,Other Non Current Liabilities,Non Current Deferred Liabilities,Non Current Deferred Taxes Liabilities,Long Term Debt And Capital Lease Obligation,Long Term Capital Lease Obligation,Long Term Debt,Current Liabilities,Other Current Liabilities,Current Deferred Liabilities,Current Deferred Revenue,Current Debt And Capital Lease Obligation,Current Debt,Other Current Borrowings,Payables And Accrued Expenses,Current Accrued Expenses,Payables,Total Tax Payable,Income Tax Payable,Accounts Payable,Total Assets,Total Non Current Assets,Other Non Current Assets,Non Current Deferred Assets,Non Current Deferred Taxes Assets,Goodwill And Other Intangible Assets,Other Intangible Assets,Goodwill,Net PPE,Accumulated Depreciation,Gross PPE,Leases,Other Properties,Machinery Furniture Equipment,Properties,Current Assets,Other Current Assets,Receivables,Taxes Receivable,Accounts Receivable,Allowance For Doubtful Accounts Receivable,Gross Accounts Receivable,Cash Cash Equivalents And Short Term Investments,Cash And Cash Equivalents,Free Cash Flow,Repurchase Of Capital Stock,Repayment Of Debt,Issuance Of Debt,Capital Expenditure,Interest Paid Supplemental Data,Income Tax Paid Supplemental Data,End Cash Position,Beginning Cash Position,Effect Of Exchange Rate Changes,Changes In Cash,Financing Cash Flow,Cash Flow From Continuing Financing Activities,Net Other Financing Charges,Proceeds From Stock Option Exercised,Net Common Stock Issuance,Common Stock Payments,Net Issuance Payments Of Debt,Net Short Term Debt Issuance,Net Long Term Debt Issuance,Long Term Debt Payments,Long Term Debt Issuance,Investing Cash Flow,Cash Flow From Continuing Investing Activities,Net Business Purchase And Sale,Sale Of Business,Purchase Of Business,Net PPE Purchase And Sale,Purchase Of PPE,Capital Expenditure Reported,Operating Cash Flow,Cash Flow From Continuing Operating Activities,Change In Working Capital,Change In Other Working Capital,Change In Other Current Assets,Change In Payables And Accrued Expense,Change In Accrued Expense,Change In Payable,Change In Account Payable,Change In Tax Payable,Change In Income Tax Payable,Change

### **2.2 Dataset description**

In [6]:
# ==================================================
# Initial Dataset Validation
# ==================================================

print(f"Number of companies: {financials_df['Symbol'].nunique():,}")

print(f"Number of sectors: {financials_df['Sector'].nunique():,}")

print(f"Number of industries: {financials_df['Industry'].nunique():,}")

Number of companies: 4,576
Number of sectors: 11
Number of industries: 145


In [7]:
financials_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4576 entries, 0 to 4575
Columns: 222 entries, Symbol to Long Term Equity Investment
dtypes: float64(217), str(5)
memory usage: 7.8 MB


In [8]:
financials_df.describe().T.head(20)

,count,mean,std,min,25%,50%,75%,max
Employees,4195.0,1.231073e+04,5.583885e+04,1.000000e+00,1.610000e+02,1.139000e+03,6.500000e+03,2.100000e+06
MarketCap,4570.0,2.108848e+10,1.729725e+11,1.308890e+05,1.516642e+08,1.159357e+09,6.347573e+09,5.189349e+12
EnterpriseValue,4512.0,7.891645e+10,2.608636e+12,-5.515685e+13,1.737587e+08,1.512113e+09,8.821250e+09,1.264616e+14
Tax Effect Of Unusual Items,4573.0,-3.105859e+08,1.463905e+10,-7.838690e+11,-1.765680e+06,0.000000e+00,0.000000e+00,2.348990e+11
Tax Rate For Calcs,4569.0,1.716604e-01,1.038284e-01,0.000000e+00,9.300000e-02,2.100000e-01,2.330000e-01,4.000000e-01
Normalized EBITDA,4048.0,3.270788e+10,1.788561e+12,-6.651988e+13,-5.521470e+06,6.408400e+07,6.144380e+08,7.343200e+13
Total Unusual Items,3626.0,-2.167490e+09,8.837795e+10,-3.919345e+12,-2.876875e+07,-1.314000e+06,1.049750e+06,8.945870e+11
Total Unusual Items Excluding Goodwill,3627.0,-2.167066e+09,8.836577e+10,-3.919345e+12,-2.878950e+07,-1.316000e+06,1.049500e+06,8.945870e+11
Net Income From Continuing Operation Net Minority Interest,4572.0,-5.395741e+09,1.513552e+12,-9.938466e+13,-2.243575e+07,1.190650e+07,2.339192e+08,1.748600e+13
Reconciled Depreciation,4472.0,2.465760e+10,6.815284e+11,-1.829000e+09,3.149193e+06,2.731600e+07,1.759378e+08,3.765300e+13


# **3. KPI Engineering**

### **3.1 Load KPI Definitions**

In [9]:
kpi_definitions_path = PROJECT_ROOT / "config" / "kpi_definitions.csv"

kpi_definitions = load_kpi_definitions(kpi_definitions_path)

print(f"Number of KPI definitions: {len(kpi_definitions)}")

kpi_definitions

Number of KPI definitions: 8


,ID,KPI,Category,UoM,What is better?,Formula,Required Fields,Include in Clustering
0,1,Gross Margin,Profitability,%,Bigger better,( `Total Revenue` - `Cost of Revenue` ) / `Tot...,Total Revenue; Cost of Revenue,Yes
1,2,Operating Margin,Profitability,%,Bigger better,`Operating Income` / `Total Revenue`,Operating Income; Total Revenue,Yes
2,3,Net Margin,Profitability,%,Bigger better,`Net Income` / `Total Revenue`,Net Income; Total Revenue,Yes
3,4,EBITDA Margin,Profitability,%,Bigger better,`EBITDA` / `Total Revenue`,EBITDA; Total Revenue,Yes
4,5,Return on Assets,Profitability,%,Bigger better,`Net Income` / `Total Assets`,Net Income; Total Assets,Yes
5,6,Return on Equity,Profitability,%,Bigger better,`Net Income` / `Stockholders Equity`,Net Income; Stockholders Equity,Yes
6,7,Asset Turnover,Profitability,%,Bigger better,`Total Revenue` / `Total Assets`,Total Revenue; Total Assets,Yes
7,8,Revenue per Employee,Profitability,USD,Bigger better,`Total Revenue` / `Employees`,Total Revenue; Employees,Yes


### **3.2 Validate KPI Definitions**

In [ ]:
kpi_validation = validate_kpi_definitions(
    data=financials_df,
    kpi_definitions=kpi_definitions
)

kpi_validation

### **3.3 Calculate KPI**

In [11]:
kpi_df = calculate_kpis(
    data=financials_df,
    kpi_definitions=kpi_definitions
)

print(f"KPI dataset shape: {kpi_df.shape}")

kpi_df.head()

ValueError: Failed to calculate KPI 'Gross Margin'. Error: name 'BACKTICK_QUOTED_STRING_Total_Revenue' is not defined